# Fine-tune the ClaimGraph comparison model on SciFact

**Optional.** The app works without this; fine-tuning teaches the model scientific wording.

On Kaggle: Settings > Accelerator > **GPU T4**, Settings > Internet > **on**, then **Save & Run All**. It should take well under an hour.

When it finishes, download `nli-scifact.zip` from the Output tab and unzip it so that this file exists:
`research-project/models/nli-scifact/config.json`. Restart the app; it uses the fine-tuned model automatically. Then run `python scripts/run.py eval` to measure it. `metrics.json` from this notebook measures the same dev split, in one direction only, so use the app's evaluation when comparing with the base model.

In [ ]:
!pip install -q -U "transformers>=4.45" "accelerate>=1.0" sentencepiece protobuf scikit-learn

In [ ]:
import json, os, pathlib, random, shutil, tarfile, urllib.request
import numpy as np, torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

BASE = 'cross-encoder/nli-deberta-v3-base'
OUT = pathlib.Path('/kaggle/working/nli-scifact') if os.path.isdir('/kaggle/working') else pathlib.Path('nli-scifact')
EPOCHS, BATCH, LR, MAX_LEN, SEED = 3, 16, 2e-5, 256, 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Turn on the GPU: Settings > Accelerator > GPU T4'
print(torch.cuda.get_device_name(0))

In [ ]:
URL = 'https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz'
folder = pathlib.Path('scifact'); folder.mkdir(exist_ok=True)
archive = folder / 'data.tar.gz'
if not archive.exists():
    urllib.request.urlretrieve(URL, archive)
with tarfile.open(archive) as tar:
    tar.extractall(folder, filter='data')

def read_jsonl(name):
    path = next(folder.rglob(name))
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

corpus = {str(doc['doc_id']): doc['abstract'] for doc in read_jsonl('corpus.jsonl')}
LABEL_MAP = {'SUPPORT': 'entailment', 'CONTRADICT': 'contradiction'}

def build(claims, neutral_per_claim=1):
    """Evidence sentences are entailment or contradiction. Other sentences of a cited abstract are neutral."""
    rows = []
    for claim in claims:
        evidence = claim.get('evidence') or {}
        used = {}
        for doc_id, groups in evidence.items():
            sentences = corpus.get(str(doc_id))
            if not sentences:
                continue
            for group in groups:
                label = LABEL_MAP.get(group.get('label'))
                if label:
                    rows.append((' '.join(sentences[i] for i in group['sentences'] if i < len(sentences)), claim['claim'], label))
                    used.setdefault(str(doc_id), set()).update(group['sentences'])
        for doc_id in claim.get('cited_doc_ids', []):
            sentences = corpus.get(str(doc_id))
            if not sentences:
                continue
            free = [i for i in range(len(sentences)) if i not in used.get(str(doc_id), set())]
            random.shuffle(free)
            for i in free[:neutral_per_claim]:
                rows.append((sentences[i], claim['claim'], 'neutral'))
    return rows

train_rows, dev_rows = build(read_jsonl('claims_train.jsonl')), build(read_jsonl('claims_dev.jsonl'))
print(len(train_rows), 'train;', len(dev_rows), 'dev')
from collections import Counter
print(Counter(row[2] for row in train_rows))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE)
label2id = {name.lower(): index for index, name in model.config.id2label.items()}
print(label2id)

class Pairs(torch.utils.data.Dataset):
    def __init__(self, rows, both_directions):
        expanded = list(rows)
        if both_directions:  # the app scores each claim pair both ways
            expanded += [(hyp, pre, label) for pre, hyp, label in rows]
        self.encodings = tokenizer([r[0] for r in expanded], [r[1] for r in expanded], truncation=True, max_length=MAX_LEN)
        self.labels = [label2id[r[2]] for r in expanded]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: value[index] for key, value in self.encodings.items()}, 'labels': self.labels[index]}

train_set, dev_set = Pairs(train_rows, True), Pairs(dev_rows, False)
counts = np.bincount(train_set.labels, minlength=3)
weights = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float)
print('class weights', weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def metrics(prediction):
    predicted = prediction.predictions.argmax(-1)
    return {'accuracy': accuracy_score(prediction.label_ids, predicted), 'macro_f1': f1_score(prediction.label_ids, predicted, average='macro')}

args = TrainingArguments(
    output_dir='checkpoints', num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
    learning_rate=LR, weight_decay=0.01, warmup_ratio=0.1, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True, fp16=True, seed=SEED, report_to='none', logging_steps=50,
)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_set, eval_dataset=dev_set, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metrics)
trainer.train()

In [ ]:
output = trainer.predict(dev_set)
predicted = output.predictions.argmax(-1)
names = [model.config.id2label[i].lower() for i in range(3)]
print(classification_report(output.label_ids, predicted, target_names=names, digits=3))

if OUT.exists():
    shutil.rmtree(OUT)
trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))
(OUT / 'metrics.json').write_text(json.dumps({'split': 'dev', 'direction': 'premise_to_hypothesis', 'n': len(dev_set), **{k: float(v) for k, v in output.metrics.items() if isinstance(v, (int, float))}}, indent=2))
shutil.make_archive(str(OUT), 'zip', OUT)
for path in OUT.parent.glob('nli-scifact*'):
    print(path)
print('Done. Download nli-scifact.zip and unzip it into research-project/models/nli-scifact/')